# **🏋🏽 Wearable Biomechanics - Annotations Manager**


In [ ]:
%load_ext autoreload
%autoreload 2

# Install OpenDatasetStore & this repo in editable mode (Colab/local)
import importlib.util, os, subprocess, sys
if importlib.util.find_spec("open_dataset_store") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/janithcyapa/OpenDatasetStore.git"])
if importlib.util.find_spec("wearable_biomechanics") is None:
    repo_dir = "/content/Wearable-Biomechanics"
    if not os.path.exists(repo_dir):
        subprocess.check_call(["git", "clone", "-b", "cleannjerk_dataset", "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", repo_dir])

from wearable_biomechanics import init_store, display_table

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
store = init_store(dataset_path=dataset_path)

In [ ]:
# @title Dataset & Trials Overview
store.summary()
display_table(store.list_entries(entry_type="complete_trials", include_paths=False))
display_table(store.list_entries(entry_type="fault_tolerant_trials", include_paths=True))

In [ ]:
# @title Interactive Annotation Editor & Saver { display-mode: "form" }
from wearable_biomechanics.annotation import launch_annotation_checker

ENTRY_TYPE = "fault_tolerant_trials" # @param ["complete_trials", "fault_tolerant_trials"]
ENTRY_ID = "fault_tolerant_trials_0015" # @param {type:"string"}
launch_annotation_checker(store=store, entry_type=ENTRY_TYPE, entry_id=ENTRY_ID)

In [ ]:
# @title 🎬 Frame-by-Frame Video Player (fast) { display-mode: "form" }
from wearable_biomechanics.annotation import render_video_player

proxy_height = 720 # @param [480, 540, 720, 1080] {type:"raw"}
render_video_player(
    store=store,
    entry_type=ENTRY_TYPE,
    entry_id=ENTRY_ID,
    label="synced_video",
    proxy_height=proxy_height,
)

# Complete Summary

In [ ]:
# @title Get Complete Summary (Publication Light Theme + Plotly & PNG Export)
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, HTML

# ==============================================================================
# CONFIGURATION
# ==============================================================================
ENTRY_TYPES = ['complete_trials', 'fault_tolerant_trials']
CANDIDATE_TAGS = [f'rep_segments_a{i}' for i in range(1, 15)]

SEGMENT_COLS = [
    'start_rest', 'setup', 'first_pull', 'second_pull', 'clean',
    'middle_rest', 'jerk_phase_one', 'jerk_phase_two', 'rest_up',
    'bring_down', 'end_rest', 'end_rep'
]

# Thresholds (at 30 fps: 8.0 frames = 266.7 ms; 5.0 frames = 166.7 ms)
STD_OUTLIER_THRESHOLD = 8.0
DEFAULT_FPS = 30.0

# ==============================================================================
# 1. EXTRACT ALL ANNOTATIONS & METADATA ACROSS ALL TRIALS
# ==============================================================================
all_annotations = []
fps_lookup = {}

for e_type in ENTRY_TYPES:
    entries_df = store.list_entries(entry_type=e_type)
    if entries_df is None or entries_df.empty:
        continue

    for _, meta_row in entries_df.iterrows():
        e_id = meta_row['entry_id']
        fps_lookup[e_id] = (
            float(meta_row['video_fps'])
            if 'video_fps' in meta_row and pd.notna(meta_row['video_fps'])
            else DEFAULT_FPS
        )

        for tag in CANDIDATE_TAGS:
            try:
                df_ann = store.get_entry_processed_data(entry_type=e_type, entry_id=e_id, tag=tag)
                if df_ann is not None and not df_ann.empty:
                    df_ann = df_ann.copy()
                    valid_mask = (df_ann[SEGMENT_COLS[1:]] != 0).any(axis=1)
                    df_ann = df_ann[valid_mask]

                    if not df_ann.empty:
                        df_ann['entry_type'] = e_type
                        df_ann['entry_id'] = e_id
                        df_ann['source_tag'] = tag
                        all_annotations.append(df_ann)
            except Exception:
                continue

raw_df = pd.concat(all_annotations, ignore_index=True)
print(f"✅ Loaded {len(raw_df)} annotated rep rows across {raw_df['entry_id'].nunique()} trials.")

# ==============================================================================
# 2. COMPUTE PER-TRIAL, PER-REP MEAN, SD (FRAMES), AND SD (MS)
# ==============================================================================
grouped = raw_df.groupby(['entry_type', 'entry_id', 'rep'], sort=True)

rep_mean_df = grouped[SEGMENT_COLS].mean()
rep_sd_df   = grouped[SEGMENT_COLS].std(ddof=1)
rep_cnt_ser = grouped['source_tag'].nunique().rename('n_annotators')

rep_sd_ms_df = rep_sd_df.copy()
for idx in rep_sd_ms_df.index:
    e_id = idx[1]
    ms_per_frame = 1000.0 / fps_lookup.get(e_id, DEFAULT_FPS)
    rep_sd_ms_df.loc[idx] = rep_sd_ms_df.loc[idx] * ms_per_frame

# ==============================================================================
# 3. PUBLICATION-READY PLOTLY HEATMAP (LIGHT THEME + 300-DPI PNG EXPORT)
# ==============================================================================
plot_sd_df   = rep_sd_df.reset_index()
plot_mean_df = rep_mean_df.reset_index()
plot_ms_df   = rep_sd_ms_df.reset_index()
plot_cnt_df  = rep_cnt_ser.reset_index()

row_labels = [
    f"{r['entry_id']}  |  Rep {int(r['rep'])} (n={int(c['n_annotators'])})"
    for (_, r), (_, c) in zip(plot_sd_df.iterrows(), plot_cnt_df.iterrows())
]

z_sd   = plot_sd_df[SEGMENT_COLS].values
z_mean = plot_mean_df[SEGMENT_COLS].values
z_ms   = plot_ms_df[SEGMENT_COLS].values

# Clamp color scale at 12 frames so 0-8f variation is clear and >8f stands out
Z_MAX_COLOR = 12.0

# Publication Colorblind-Safe Sequential Palette (Light Ice -> Steel Blue -> Amber -> Brick Red)
pub_colorscale = [
    [0.00, "#f0f9ff"],  # 0.0f: Crisp Ice White-Blue (Near-zero variance)
    [0.21, "#bae6fd"],  # ~2.5f: Soft Sky Blue (Gold Standard <= 2.5f)
    [0.42, "#7dd3fc"],  # ~5.0f: Medium Cerulean (Good <= 5.0f)
    [0.66, "#fde68a"],  # ~8.0f: Muted Ochre/Amber (Acceptable 5-8f)
    [0.67, "#fca5a5"],  # >8.0f: Soft Coral Rose (Outlier threshold crossed)
    [1.00, "#b91c1c"]   # >=12f: Deep Academic Brick Crimson (Severe outlier)
]

hover_text = []
cell_text = []
for i in range(len(row_labels)):
    h_row, c_row = [], []
    e_id = plot_sd_df.loc[i, 'entry_id']
    rep_n = int(plot_sd_df.loc[i, 'rep'])
    n_ann = int(plot_cnt_df.loc[i, 'n_annotators'])
    for j, col in enumerate(SEGMENT_COLS):
        sd_v = z_sd[i, j]
        mn_v = z_mean[i, j]
        ms_v = z_ms[i, j]
        if pd.isna(sd_v):
            c_row.append("–")
            h_row.append(
                f"<b>{e_id} | Rep {rep_n}</b><br>"
                f"Phase: <b>{col}</b><br>"
                f"Mean: {mn_v:.1f} f<br>"
                f"SD: N/A (n={n_ann})"
            )
        else:
            c_row.append(f"{sd_v:.1f}")
            status = (
                "Severe (>30f / >1000ms)" if sd_v > 30 else
                "Outlier (>8f / >267ms)" if sd_v > STD_OUTLIER_THRESHOLD else
                "Acceptable (5–8f / 167–267ms)" if sd_v > 5.0 else
                "Good (2.5–5f / 83–167ms)" if sd_v > 2.5 else
                "High Agreement (≤2.5f / ≤83ms)"
            )
            h_row.append(
                f"<b>{e_id} | Rep {rep_n}</b> (n={n_ann})<br>"
                f"Phase: <b>{col}</b><br>"
                f"Mean Frame: <b>{mn_v:.1f} f</b><br>"
                f"SD (Frames): <b>{sd_v:.2f} f</b><br>"
                f"SD (Time): <b>{ms_v:.0f} ms</b><br>"
                f"Tier: <b>{status}</b>"
            )
    hover_text.append(h_row)
    cell_text.append(c_row)

fig_height = max(650, len(row_labels) * 26 + 160)

fig = go.Figure(data=go.Heatmap(
    z=np.nan_to_num(z_sd, nan=-0.5),
    x=SEGMENT_COLS,
    y=row_labels,
    text=cell_text,
    texttemplate="%{text}",
    textfont={"size": 10.5, "family": "Arial, Helvetica, sans-serif", "color": "#111827"},
    hoverinfo="text",
    hovertext=hover_text,
    colorscale=pub_colorscale,
    zmin=0,
    zmax=Z_MAX_COLOR,
    xgap=1.5,
    ygap=1.5,
    colorbar=dict(
        title=dict(text="<b>SD (Frames)</b>", side="right", font=dict(color="#111827", size=12)),
        tickvals=[0, 2.5, 5.0, 8.0, 12.0],
        ticktext=["0.0 f (0 ms)", "2.5 f (83 ms)", "5.0 f (167 ms)", "8.0 f (267 ms)", "≥12.0 f (Outlier)"],
        tickfont=dict(color="#374151", size=11),
        len=0.45,
        thickness=15,
        outlinecolor="#9ca3af",
        outlinewidth=1
    )
))

# Crisp dark charcoal horizontal lines separating distinct trials
trial_ids_ordered = plot_sd_df['entry_id'].tolist()
shapes = []
for row_idx in range(1, len(trial_ids_ordered)):
    if trial_ids_ordered[row_idx] != trial_ids_ordered[row_idx - 1]:
        shapes.append(dict(
            type="line",
            xref="paper", x0=0, x1=1,
            yref="y", y0=row_idx - 0.5, y1=row_idx - 0.5,
            line=dict(color="#1f2937", width=2.0)
        ))

fig.update_layout(
    title=dict(
        text="<b>Inter-Annotator Temporal Agreement Across Movement Phases</b><br>"
             "<span style='font-size:12px;color:#4b5563;'>Per-Trial & Per-Repetition Standard Deviation in Frames (Threshold = 8.0 frames / 267 ms at 30 Hz)</span>",
        font=dict(size=15, color="#111827", family="Arial, Helvetica, sans-serif"),
        x=0.01,
        y=0.985
    ),
    height=fig_height,
    width=1320,
    paper_bgcolor="#ffffff",
    plot_bgcolor="#e5e7eb",
    font=dict(family="Arial, Helvetica, sans-serif", color="#111827"),
    xaxis=dict(
        title=dict(text="<b>Annotated Movement Phase</b>", font=dict(size=12, color="#111827")),
        side="top",
        tickangle=-25,
        tickfont=dict(size=11, color="#1f2937"),
        showline=True,
        linewidth=1,
        linecolor="#9ca3af",
        mirror=True
    ),
    yaxis=dict(
        title=dict(text="<b>Trial ID  |  Repetition (Annotator Count)</b>", font=dict(size=12, color="#111827")),
        autorange="reversed",
        tickfont=dict(size=10.5, color="#1f2937"),
        showline=True,
        linewidth=1,
        linecolor="#9ca3af",
        mirror=True
    ),
    shapes=shapes,
    margin=dict(l=240, r=130, t=130, b=40)
)

# High-DPI (3x scale) PNG & SVG export options in toolbar
fig.show(config={
    'displaylogo': False,
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'figure_inter_annotator_heatmap_pub',
        'height': fig_height,
        'width': 1320,
        'scale': 3
    }
})

# ==============================================================================
# 4. PUBLICATION-GRADE LIGHT HTML TABLE WITH 1-CLICK PNG EXPORT
# ==============================================================================
def get_pub_cell_style(sd_val):
    if pd.isna(sd_val):
        return "background:#f3f4f6; color:#9ca3af;"
    elif sd_val > 30.0:
        # Severe (>30f): Deep Brick Red fill with crisp white text
        return "background:#991b1b; color:#ffffff; font-weight:700;"
    elif sd_val > STD_OUTLIER_THRESHOLD:
        # Outlier (>8f): Soft Rose fill with dark crimson text
        return "background:#fee2e2; color:#991b1b; font-weight:700;"
    elif sd_val > 5.0:
        # Acceptable (5-8f): Soft Warm Amber fill with dark bronze text
        return "background:#fef3c7; color:#92400e; font-weight:600;"
    elif sd_val > 2.5:
        # Good (2.5-5f): Soft Sky Blue fill with deep navy text
        return "background:#e0f2fe; color:#075985;"
    else:
        # High Agreement (<=2.5f): Subtle Ice-Teal tint with slate text
        return "background:#f0fdf4; color:#166534;"

html_parts = ["""
<script src="https://cdnjs.cloudflare.com/ajax/libs/html2canvas/1.4.1/html2canvas.min.js"></script>
<script>
function downloadPubTablePNG() {
    const btn = document.getElementById('dl-pub-png-btn');
    const container = document.getElementById('exportable-pub-container');
    const scrollWrap = document.getElementById('pub-scroll-wrap');

    btn.innerText = 'Rendering 300-DPI PNG...';
    btn.disabled = true;

    const origMaxHeight = scrollWrap.style.maxHeight;
    const origOverflow = scrollWrap.style.overflow;
    scrollWrap.style.maxHeight = 'none';
    scrollWrap.style.overflow = 'visible';

    html2canvas(container, {
        backgroundColor: '#ffffff',
        scale: 2.5,
        useCORS: true
    }).then(canvas => {
        scrollWrap.style.maxHeight = origMaxHeight;
        scrollWrap.style.overflow = origOverflow;

        const link = document.createElement('a');
        link.download = 'table_per_rep_annotation_summary.png';
        link.href = canvas.toDataURL('image/png');
        link.click();

        btn.innerText = '⬇ Download Publication Table (PNG)';
        btn.disabled = false;
    }).catch(err => {
        console.error(err);
        scrollWrap.style.maxHeight = origMaxHeight;
        scrollWrap.style.overflow = origOverflow;
        btn.innerText = 'Export Failed (See Console)';
        btn.disabled = false;
    });
}
</script>

<style>
    .pub-outer {
        background: #ffffff;
        padding: 20px;
        border-radius: 6px;
        border: 1px solid #d1d5db;
        font-family: 'Arial', 'Helvetica Neue', sans-serif;
        color: #111827;
        margin-top: 20px;
        box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);
    }
    .pub-toolbar {
        display: flex;
        justify-content: space-between;
        align-items: flex-start;
        flex-wrap: wrap;
        gap: 12px;
        margin-bottom: 14px;
        padding-bottom: 12px;
        border-bottom: 2px solid #111827;
    }
    .pub-legend {
        display: flex;
        gap: 8px;
        flex-wrap: wrap;
        align-items: center;
        font-size: 12px;
        color: #374151;
        margin-top: 6px;
    }
    .pub-badge {
        padding: 2px 8px;
        border-radius: 4px;
        font-weight: 600;
        font-size: 11px;
        border: 1px solid rgba(0,0,0,0.12);
    }
    .pub-dl-btn {
        background: #111827;
        color: #ffffff;
        border: 1px solid #111827;
        padding: 7px 14px;
        border-radius: 4px;
        font-size: 12px;
        font-weight: 600;
        cursor: pointer;
        transition: background 0.15s ease;
    }
    .pub-dl-btn:hover {
        background: #374151;
    }
    .pub-grid-wrap {
        max-height: 750px;
        overflow: auto;
        border: 1px solid #9ca3af;
        background: #ffffff;
    }
    .pub-table {
        border-collapse: collapse;
        width: max-content;
        font-size: 11.5px;
        font-variant-numeric: tabular-nums;
    }
    .pub-table th {
        position: sticky;
        top: 0;
        background: #f8fafc;
        color: #111827;
        padding: 8px 10px;
        text-align: right;
        border-top: 2px solid #111827;
        border-bottom: 2px solid #111827;
        border-left: 1px solid #e5e7eb;
        z-index: 2;
        font-weight: 700;
    }
    .pub-table th:first-child {
        text-align: left;
        min-width: 210px;
        border-left: none;
    }
    .pub-trial-header td {
        background: #e2e8f0;
        color: #0f172a;
        font-weight: 700;
        font-size: 12px;
        padding: 7px 10px;
        border-top: 2px solid #475569;
        border-bottom: 1px solid #94a3b8;
    }
    .pub-rep-divider td {
        border-bottom: 1.5px solid #cbd5e1;
    }
    .pub-metric-label {
        color: #1f2937;
        padding: 4px 10px;
        white-space: nowrap;
        background: #ffffff;
        border-right: 1px solid #cbd5e1;
    }
    .pub-num-cell {
        text-align: right;
        padding: 4px 8px;
        min-width: 76px;
        border-left: 1px solid #e5e7eb;
        border-bottom: 1px solid #f1f5f9;
    }
</style>

<div class="pub-outer" id="exportable-pub-container">
    <div class="pub-toolbar">
        <div>
            <h3 style="margin:0; font-size:15px; color:#111827; text-transform:uppercase; letter-spacing:0.3px;">
                Table 1. Per-Trial and Per-Repetition Inter-Annotator Summary (Mean, SD in Frames, SD in ms)
            </h3>
            <div class="pub-legend">
                <span><b>SD Agreement Classification:</b></span>
                <span class="pub-badge" style="background:#f0fdf4; color:#166534;">High (&le;2.5f / &le;83ms)</span>
                <span class="pub-badge" style="background:#e0f2fe; color:#075985;">Good (2.5–5f / 83–167ms)</span>
                <span class="pub-badge" style="background:#fef3c7; color:#92400e;">Moderate (5–8f / 167–267ms)</span>
                <span class="pub-badge" style="background:#fee2e2; color:#991b1b;">Outlier (&gt;8f / &gt;267ms)</span>
                <span class="pub-badge" style="background:#991b1b; color:#ffffff;">Severe (&gt;30f / &gt;1000ms)</span>
            </div>
        </div>
        <button id="dl-pub-png-btn" class="pub-dl-btn" onclick="downloadPubTablePNG()">⬇ Download Publication Table (PNG)</button>
    </div>

    <div class="pub-grid-wrap" id="pub-scroll-wrap">
    <table class="pub-table">
    <thead>
        <tr>
            <th>Trial / Repetition / Statistic</th>
"""]

for col in SEGMENT_COLS:
    html_parts.append(f"<th>{col}</th>")
html_parts.append("</tr></thead><tbody>")

for (e_type, e_id), trial_group in rep_sd_df.groupby(level=[0, 1], sort=True):
    fps_val = fps_lookup.get(e_id, DEFAULT_FPS)
    n_reps = len(trial_group)
    n_ann_max = int(rep_cnt_ser.loc[(e_type, e_id)].max())
    cohort_label = "Complete Cohort" if e_type == "complete_trials" else "Fault-Tolerant Cohort"

    html_parts.append(
        f"<tr class='pub-trial-header'><td colspan='{len(SEGMENT_COLS) + 1}'>"
        f"[{cohort_label}] &nbsp;<b>{e_id}</b> &nbsp;&bull;&nbsp; {n_reps} Repetitions &nbsp;&bull;&nbsp; "
        f"{n_ann_max} Annotators &nbsp;&bull;&nbsp; {int(fps_val)} Hz"
        f"</td></tr>"
    )

    for (_, _, rep_num) in trial_group.index:
        means = rep_mean_df.loc[(e_type, e_id, rep_num)]
        sds   = rep_sd_df.loc[(e_type, e_id, rep_num)]
        sd_ms = rep_sd_ms_df.loc[(e_type, e_id, rep_num)]
        n_ann = int(rep_cnt_ser.loc[(e_type, e_id, rep_num)])

        # Row 1: Mean (Frames)
        html_parts.append(f"<tr><td class='pub-metric-label'><b>Rep {rep_num}</b> <span style='color:#6b7280;'>(n={n_ann})</span> &mdash; Mean (frames)</td>")
        for col in SEGMENT_COLS:
            m_val = means[col]
            html_parts.append(f"<td class='pub-num-cell' style='background:#ffffff; color:#374151;'>{m_val:.1f}</td>")
        html_parts.append("</tr>")

        # Row 2: SD (Frames)
        html_parts.append(f"<tr><td class='pub-metric-label' style='color:#374151;'>&nbsp;&nbsp;&lfloor; SD (frames)</td>")
        for col in SEGMENT_COLS:
            s_val = sds[col]
            c_style = get_pub_cell_style(s_val)
            txt = f"{s_val:.1f}" if pd.notna(s_val) else "–"
            html_parts.append(f"<td class='pub-num-cell' style='{c_style}'>{txt}</td>")
        html_parts.append("</tr>")

        # Row 3: SD (ms)
        html_parts.append(f"<tr class='pub-rep-divider'><td class='pub-metric-label' style='color:#6b7280;'>&nbsp;&nbsp;&lfloor; SD (ms)</td>")
        for col in SEGMENT_COLS:
            s_val = sds[col]
            ms_val = sd_ms[col]
            c_style = get_pub_cell_style(s_val)
            txt = f"{ms_val:.0f} ms" if pd.notna(ms_val) else "–"
            html_parts.append(f"<td class='pub-num-cell' style='{c_style}'>{txt}</td>")
        html_parts.append("</tr>")

html_parts.append("</tbody></table></div></div>")
display(HTML("".join(html_parts)))

In [16]:
# @title Get Unified Overall Summary (All Trials Pooled — Distributions, Plots & Tables)
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# ==============================================================================
# CONFIGURATION
# ==============================================================================
ENTRY_TYPES = ['complete_trials', 'fault_tolerant_trials']
CANDIDATE_TAGS = [f'rep_segments_a{i}' for i in range(1, 15)]

SEGMENT_COLS = [
    'start_rest', 'setup', 'first_pull', 'second_pull', 'clean',
    'middle_rest', 'jerk_phase_one', 'jerk_phase_two', 'rest_up',
    'bring_down', 'end_rest', 'end_rep'
]

STD_OUTLIER_THRESHOLD = 8.0  # >8.0 frames (>267 ms at 30 Hz)
SD_THRESHOLDS = [0, 1, 2, 3, 4, 5, 8]  # Cumulative cutoffs for Table A
DEFAULT_FPS = 30.0

# ==============================================================================
# 1. EXTRACT & POOL ALL ANNOTATIONS (IGNORING COHORT SEPARATION)
# ==============================================================================
if 'raw_df' not in globals() or raw_df is None or raw_df.empty:
    all_annotations = []
    fps_lookup = {}
    for e_type in ENTRY_TYPES:
        entries_df = store.list_entries(entry_type=e_type)
        if entries_df is None or entries_df.empty:
            continue
        for _, meta_row in entries_df.iterrows():
            e_id = meta_row['entry_id']
            fps_lookup[e_id] = (
                float(meta_row['video_fps'])
                if 'video_fps' in meta_row and pd.notna(meta_row['video_fps'])
                else DEFAULT_FPS
            )
            for tag in CANDIDATE_TAGS:
                try:
                    df_ann = store.get_entry_processed_data(entry_type=e_type, entry_id=e_id, tag=tag)
                    if df_ann is not None and not df_ann.empty:
                        df_ann = df_ann.copy()
                        valid_mask = (df_ann[SEGMENT_COLS[1:]] != 0).any(axis=1)
                        df_ann = df_ann[valid_mask]
                        if not df_ann.empty:
                            df_ann['entry_id'] = e_id
                            df_ann['source_tag'] = tag
                            all_annotations.append(df_ann)
                except Exception:
                    continue
    raw_df = pd.concat(all_annotations, ignore_index=True)
else:
    fps_lookup = globals().get('fps_lookup', {})

# Group strictly by (entry_id, rep) — no separation by entry_type
grouped_pooled = raw_df.groupby(['entry_id', 'rep'], sort=True)

pooled_mean_df = grouped_pooled[SEGMENT_COLS].mean()
pooled_sd_df   = grouped_pooled[SEGMENT_COLS].std(ddof=1)
pooled_cnt_ser = grouped_pooled['source_tag'].nunique().rename('n_annotators')

# Convert SD to milliseconds using each trial's FPS
pooled_sd_ms_df = pooled_sd_df.copy()
for (e_id, rep_num) in pooled_sd_ms_df.index:
    ms_per_frame = 1000.0 / fps_lookup.get(e_id, DEFAULT_FPS)
    pooled_sd_ms_df.loc[(e_id, rep_num)] = pooled_sd_ms_df.loc[(e_id, rep_num)] * ms_per_frame

total_trials = raw_df['entry_id'].nunique()
total_unique_reps = len(pooled_sd_df)
total_annotations = len(raw_df)

print(
    f"✅ Pooled Dataset: {total_trials} Total Trials | "
    f"{total_unique_reps} Unique Repetitions | "
    f"{total_annotations} Total Annotator Rows"
)

# ==============================================================================
# 2. COMPUTE GRAND OVERALL PHASE-LEVEL SUMMARY STATISTICS
# ==============================================================================
phase_stats = []
for col in SEGMENT_COLS:
    s_f  = pooled_sd_df[col].dropna()
    s_ms = pooled_sd_ms_df[col].dropna()
    m_f  = pooled_mean_df[col].dropna()

    phase_stats.append({
        'phase': col,
        'n_reps': len(s_f),
        'grand_mean_frame': m_f.mean(),
        'mean_sd_f': s_f.mean(),
        'sd_sd_f': s_f.std(ddof=1),
        'med_sd_f': s_f.median(),
        'q25_f': s_f.quantile(0.25),
        'q75_f': s_f.quantile(0.75),
        'iqr_f': s_f.quantile(0.75) - s_f.quantile(0.25),
        'p95_f': s_f.quantile(0.95),
        'max_f': s_f.max(),
        'mean_sd_ms': s_ms.mean(),
        'med_sd_ms': s_ms.median(),
        'iqr_ms': s_ms.quantile(0.75) - s_ms.quantile(0.25),
        'pct_high': (s_f <= 2.5).mean() * 100.0,
        'pct_good': ((s_f > 2.5) & (s_f <= 5.0)).mean() * 100.0,
        'pct_mod':  ((s_f > 5.0) & (s_f <= STD_OUTLIER_THRESHOLD)).mean() * 100.0,
        'pct_pass': (s_f <= STD_OUTLIER_THRESHOLD).mean() * 100.0,
        'pct_out':  (s_f > STD_OUTLIER_THRESHOLD).mean() * 100.0
    })

phase_summary_df = pd.DataFrame(phase_stats)

# Dataset-wide flattened arrays across all phases & reps
all_sd_f_flat  = pooled_sd_df[SEGMENT_COLS].values.flatten()
all_sd_f_flat  = all_sd_f_flat[~np.isnan(all_sd_f_flat)]
all_sd_ms_flat = pooled_sd_ms_df[SEGMENT_COLS].values.flatten()
all_sd_ms_flat = all_sd_ms_flat[~np.isnan(all_sd_ms_flat)]

# ==============================================================================
# 3. PLOT 1: INTER-ANNOTATOR SD DISTRIBUTION (CUMULATIVE CDF + FRAME BINS)
# ==============================================================================
fig_dist = make_subplots(
    rows=1, cols=2,
    column_widths=[0.48, 0.52],
    subplot_titles=(
        "<b>A. Cumulative Agreement Curve (CDF: % Within ≤ x Frames)</b>",
        "<b>B. Per-Phase SD Distribution by Frame Bin (%)</b>"
    ),
    horizontal_spacing=0.09
)

x_grid = np.linspace(0, 10.0, 201)
phase_colors = px.colors.qualitative.Dark24

for idx, col in enumerate(SEGMENT_COLS):
    s_f = pooled_sd_df[col].dropna().values
    cdf_vals = [(s_f <= x).mean() * 100.0 for x in x_grid]
    fig_dist.add_trace(
        go.Scatter(
            x=x_grid,
            y=cdf_vals,
            mode="lines",
            name=col,
            line=dict(color=phase_colors[idx % len(phase_colors)], width=1.6),
            opacity=0.75,
            legendgroup="phases",
            legendgrouptitle=dict(text="<b>Phases (Panel A)</b>"),
            hovertemplate=f"<b>{col}</b><br>Threshold: ≤ %{{x:.1f}} f<br>Within: <b>%{{y:.1f}}%</b><extra></extra>"
        ),
        row=1, col=1
    )

grand_cdf = [(all_sd_f_flat <= x).mean() * 100.0 for x in x_grid]
fig_dist.add_trace(
    go.Scatter(
        x=x_grid,
        y=grand_cdf,
        mode="lines",
        name="All Phases Combined",
        line=dict(color="#111827", width=3.5),
        legendgroup="phases",
        hovertemplate="<b>All Phases Combined</b><br>Threshold: ≤ %{x:.1f} f<br>Within: <b>%{y:.1f}%</b><extra></extra>"
    ),
    row=1, col=1
)

for ref_f, ref_label, ref_col in [
    (2.0, "2f (67ms)", "#0284c7"),
    (5.0, "5f (167ms)", "#d97706"),
    (8.0, "8f (267ms)", "#b91c1c"),
]:
    fig_dist.add_vline(
        x=ref_f, line_dash="dot", line_color=ref_col, line_width=1.3,
        annotation_text=ref_label, annotation_position="bottom right",
        annotation_font=dict(size=9.5, color=ref_col),
        row=1, col=1
    )

bin_defs = [
    ("= 0f", lambda s: s == 0,             "#15803d"),
    ("0–1f", lambda s: (s > 0) & (s <= 1), "#4ade80"),
    ("1–2f", lambda s: (s > 1) & (s <= 2), "#86efac"),
    ("2–3f", lambda s: (s > 2) & (s <= 3), "#7dd3fc"),
    ("3–5f", lambda s: (s > 3) & (s <= 5), "#38bdf8"),
    ("5–8f", lambda s: (s > 5) & (s <= 8), "#fde047"),
    ("> 8f", lambda s: s > 8,              "#f87171"),
]

x_labels_b = SEGMENT_COLS + ["ALL PHASES"]

for b_label, cond_fn, b_color in bin_defs:
    y_vals = [cond_fn(pooled_sd_df[col].dropna()).mean() * 100.0 for col in SEGMENT_COLS]
    y_vals.append(cond_fn(all_sd_f_flat).mean() * 100.0)

    fig_dist.add_trace(
        go.Bar(
            x=x_labels_b,
            y=y_vals,
            name=b_label,
            marker=dict(color=b_color, line=dict(color="#ffffff", width=0.6)),
            legendgroup="bins",
            legendgrouptitle=dict(text="<b>SD Bins (Panel B)</b>"),
            hovertemplate=f"<b>%{{x}}</b><br>Bin {b_label}: <b>%{{y:.1f}}%</b><extra></extra>"
        ),
        row=1, col=2
    )

fig_dist.update_layout(
    title=dict(
        text=f"<b>Inter-Annotator SD Distribution Across Movement Phases ({total_trials} Trials, {total_unique_reps} Reps Pooled)</b><br>"
             f"<span style='font-size:12px;color:#4b5563;'>Cumulative agreement curves (left) and exact frame-bin proportions (right) • Click Camera Icon for 300-DPI PNG</span>",
        font=dict(size=15, color="#111827", family="Arial, Helvetica, sans-serif"),
        x=0.01
    ),
    barmode="stack",
    height=560,
    width=1360,
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    font=dict(family="Arial, Helvetica, sans-serif", color="#111827"),
    legend=dict(
        x=1.02, y=1.0, xanchor="left", yanchor="top",
        font=dict(size=10.5), tracegroupgap=14,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#e5e7eb", borderwidth=1
    ),
    margin=dict(l=60, r=190, t=100, b=90)
)

fig_dist.update_xaxes(
    title_text="<b>Inter-Annotator SD Threshold (Frames)</b>",
    range=[0, 10], dtick=1,
    showline=True, linecolor="#9ca3af", mirror=True, gridcolor="#e5e7eb",
    row=1, col=1
)
fig_dist.update_yaxes(
    title_text="<b>Cumulative Repetitions Within Threshold (%)</b>",
    range=[0, 102], dtick=10,
    showline=True, linecolor="#9ca3af", mirror=True, gridcolor="#e5e7eb",
    row=1, col=1
)
fig_dist.update_xaxes(
    tickangle=-35,
    showline=True, linecolor="#9ca3af", mirror=True, gridcolor="#f1f5f9",
    row=1, col=2
)
fig_dist.update_yaxes(
    title_text="<b>Percentage of Repetitions (%)</b>",
    range=[0, 100], dtick=10,
    showline=True, linecolor="#9ca3af", mirror=True, gridcolor="#e5e7eb",
    row=1, col=2
)

display(HTML(fig_dist.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    config={
        'displaylogo': False,
        'toImageButtonOptions': {
            'format': 'png',
            'filename': 'figure_sd_distribution_cdf_and_bins',
            'height': 560,
            'width': 1360,
            'scale': 3
        }
    }
)))

# ==============================================================================
# 4. PLOT 2: GRAND OVERALL PHASE SUMMARY (BAR + MEDIAN/IQR + AGREEMENT %)
# ==============================================================================
fig_overall = make_subplots(
    rows=1, cols=2,
    column_widths=[0.58, 0.42],
    subplot_titles=(
        "<b>A. Inter-Annotator Disagreement by Phase (Mean vs. Median ± IQR)</b>",
        "<b>B. Proportion of Repetitions by Agreement Tier (%)</b>"
    ),
    horizontal_spacing=0.10
)

fig_overall.add_trace(
    go.Bar(
        x=phase_summary_df['phase'],
        y=phase_summary_df['mean_sd_f'],
        name="Mean SD (Frames)",
        marker=dict(color="#bae6fd", line=dict(color="#0284c7", width=1.2)),
        text=[f"{v:.2f}f" for v in phase_summary_df['mean_sd_f']],
        textposition="outside",
        textfont=dict(size=10, color="#0f172a"),
        hovertemplate="<b>%{x}</b><br>Mean SD: %{y:.2f} frames<extra></extra>"
    ),
    row=1, col=1
)

fig_overall.add_trace(
    go.Scatter(
        x=phase_summary_df['phase'],
        y=phase_summary_df['med_sd_f'],
        mode="markers",
        name="Median SD (IQR Error Bars)",
        marker=dict(color="#1e3a8a", size=8, symbol="diamond"),
        error_y=dict(
            type="data",
            symmetric=False,
            array=phase_summary_df['q75_f'] - phase_summary_df['med_sd_f'],
            arrayminus=phase_summary_df['med_sd_f'] - phase_summary_df['q25_f'],
            color="#1e3a8a",
            thickness=1.5,
            width=4
        ),
        hovertemplate="<b>%{x}</b><br>Median SD: %{y:.2f} frames<extra></extra>"
    ),
    row=1, col=1
)

fig_overall.add_hline(
    y=STD_OUTLIER_THRESHOLD, line_dash="dash", line_color="#b91c1c", line_width=1.5,
    annotation_text="Outlier Threshold (8.0f / 267ms)",
    annotation_position="top left",
    annotation_font=dict(size=10, color="#b91c1c"),
    row=1, col=1
)

tier_configs = [
    ("pct_high", "High (≤2.5f)", "#86efac", "#166534"),
    ("pct_good", "Good (2.5–5f)", "#7dd3fc", "#075985"),
    ("pct_mod",  "Moderate (5–8f)", "#fde68a", "#92400e"),
    ("pct_out",  "Outlier (>8f)", "#fca5a5", "#991b1b")
]

for col_key, label, fill_c, border_c in tier_configs:
    fig_overall.add_trace(
        go.Bar(
            x=phase_summary_df['phase'],
            y=phase_summary_df[col_key],
            name=label,
            marker=dict(color=fill_c, line=dict(color=border_c, width=0.8)),
            hovertemplate=f"<b>%{{x}}</b><br>{label}: %{{y:.1f}}%<extra></extra>"
        ),
        row=1, col=2
    )

fig_overall.update_layout(
    title=dict(
        text=f"<b>Grand Overall Inter-Annotator Segmentation Summary ({total_trials} Trials, {total_unique_reps} Repetitions Pooled)</b><br>"
             f"<span style='font-size:12px;color:#4b5563;'>All trials combined without cohort separation • Click Camera Icon in Toolbar for 300-DPI PNG</span>",
        font=dict(size=15, color="#111827", family="Arial, Helvetica, sans-serif"),
        x=0.01
    ),
    barmode="stack",
    height=500,
    width=1320,
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    font=dict(family="Arial, Helvetica, sans-serif", color="#111827"),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.08, xanchor="right", x=1.0,
        font=dict(size=11), bgcolor="rgba(255,255,255,0.9)"
    ),
    margin=dict(l=60, r=40, t=115, b=85)
)

fig_overall.update_xaxes(tickangle=-30, showline=True, linecolor="#9ca3af", mirror=True, gridcolor="#f1f5f9")
fig_overall.update_yaxes(
    title_text="<b>Inter-Annotator SD (Frames)</b>",
    showline=True, linecolor="#9ca3af", mirror=True, gridcolor="#e5e7eb", row=1, col=1
)
fig_overall.update_yaxes(
    title_text="<b>Percentage of Repetitions (%)</b>", range=[0, 100],
    showline=True, linecolor="#9ca3af", mirror=True, gridcolor="#e5e7eb", row=1, col=2
)

display(HTML(fig_overall.to_html(
    include_plotlyjs=False,
    full_html=False,
    config={
        'displaylogo': False,
        'toImageButtonOptions': {
            'format': 'png',
            'filename': 'figure_grand_overall_phase_summary',
            'height': 500,
            'width': 1320,
            'scale': 3
        }
    }
)))

# ==============================================================================
# 5. PLOT 3: TRIAL-LEVEL POOLED HEATMAP (1 ROW PER TRIAL, ALL REPS AVERAGED)
# ==============================================================================
trial_mean_sd_df = pooled_sd_df.groupby(level=0).mean()
trial_mean_ms_df = pooled_sd_ms_df.groupby(level=0).mean()
trial_rep_counts = pooled_sd_df.groupby(level=0).size()
trial_ann_counts = raw_df.groupby('entry_id')['source_tag'].nunique()

trial_labels = [
    f"{e_id}  |  {int(trial_rep_counts[e_id])} Reps (n={int(trial_ann_counts[e_id])})"
    for e_id in trial_mean_sd_df.index
]

z_trial_sd = trial_mean_sd_df[SEGMENT_COLS].values
z_trial_ms = trial_mean_ms_df[SEGMENT_COLS].values

pub_colorscale = [
    [0.00, "#f0f9ff"],
    [0.21, "#bae6fd"],
    [0.42, "#7dd3fc"],
    [0.66, "#fde68a"],
    [0.67, "#fca5a5"],
    [1.00, "#b91c1c"]
]

hover_trial, text_trial = [], []
for i, e_id in enumerate(trial_mean_sd_df.index):
    h_row, c_row = [], []
    for j, col in enumerate(SEGMENT_COLS):
        sv = z_trial_sd[i, j]
        mv = z_trial_ms[i, j]
        if pd.isna(sv):
            c_row.append("–")
            h_row.append(f"<b>{e_id}</b><br>Phase: {col}<br>Mean SD: N/A")
        else:
            c_row.append(f"{sv:.1f}")
            h_row.append(
                f"<b>{e_id}</b> ({int(trial_rep_counts[e_id])} Reps)<br>"
                f"Phase: <b>{col}</b><br>"
                f"Trial Mean SD: <b>{sv:.2f} f</b> ({mv:.0f} ms)"
            )
    hover_trial.append(h_row)
    text_trial.append(c_row)

fig_trial_h = max(550, len(trial_labels) * 22 + 150)
fig_trials = go.Figure(data=go.Heatmap(
    z=np.nan_to_num(z_trial_sd, nan=-0.5),
    x=SEGMENT_COLS,
    y=trial_labels,
    text=text_trial,
    texttemplate="%{text}",
    textfont={"size": 10.5, "family": "Arial, Helvetica, sans-serif", "color": "#111827"},
    hoverinfo="text",
    hovertext=hover_trial,
    colorscale=pub_colorscale,
    zmin=0,
    zmax=12.0,
    xgap=1.5,
    ygap=1.5,
    colorbar=dict(
        title=dict(text="<b>Mean SD (f)</b>", side="right", font=dict(color="#111827", size=12)),
        tickvals=[0, 2.5, 5.0, 8.0, 12.0],
        ticktext=["0.0 f", "2.5 f", "5.0 f", "8.0 f", "≥12.0 f"],
        len=0.5,
        thickness=15,
        outlinecolor="#9ca3af",
        outlinewidth=1
    )
))

fig_trials.update_layout(
    title=dict(
        text=f"<b>Unified Per-Trial Inter-Annotator Summary Heatmap ({total_trials} Trials Pooled, Averaged Across Reps)</b><br>"
             f"<span style='font-size:12px;color:#4b5563;'>Each row represents a single trial (all repetitions pooled, sorted by Trial ID)</span>",
        font=dict(size=15, color="#111827", family="Arial, Helvetica, sans-serif"),
        x=0.01
    ),
    height=fig_trial_h,
    width=1320,
    paper_bgcolor="#ffffff",
    plot_bgcolor="#e5e7eb",
    font=dict(family="Arial, Helvetica, sans-serif", color="#111827"),
    xaxis=dict(
        title=dict(text="<b>Annotated Movement Phase</b>", font=dict(size=12)),
        side="top", tickangle=-25, showline=True, linewidth=1, linecolor="#9ca3af", mirror=True
    ),
    yaxis=dict(
        title=dict(text="<b>Trial ID  |  Total Reps (Annotators)</b>", font=dict(size=12)),
        autorange="reversed", showline=True, linewidth=1, linecolor="#9ca3af", mirror=True
    ),
    margin=dict(l=230, r=120, t=125, b=40)
)

display(HTML(fig_trials.to_html(
    include_plotlyjs=False,
    full_html=False,
    config={
        'displaylogo': False,
        'toImageButtonOptions': {
            'format': 'png',
            'filename': 'figure_unified_per_trial_heatmap',
            'height': fig_trial_h,
            'width': 1320,
            'scale': 3
        }
    }
)))

# ==============================================================================
# 6. PUBLICATION-GRADE HTML SUMMARY TABLES (SD DISTRIBUTION + POOLED TRIAL-LEVEL)
# ==============================================================================
def get_pub_cell_style(sd_val):
    if pd.isna(sd_val):
        return "background:#f3f4f6; color:#9ca3af;"
    elif sd_val > 30.0:
        return "background:#991b1b; color:#ffffff; font-weight:700;"
    elif sd_val > STD_OUTLIER_THRESHOLD:
        return "background:#fee2e2; color:#991b1b; font-weight:700;"
    elif sd_val > 5.0:
        return "background:#fef3c7; color:#92400e; font-weight:600;"
    elif sd_val > 2.5:
        return "background:#e0f2fe; color:#075985;"
    else:
        return "background:#f0fdf4; color:#166534;"

def get_pct_cell_style(pct, invert=False):
    if invert:
        if pct >= 15.0: return "background:#fee2e2; color:#991b1b; font-weight:700;"
        if pct >= 5.0:  return "background:#fef3c7; color:#92400e; font-weight:600;"
        return "background:#f0fdf4; color:#166534;"
    else:
        if pct >= 90.0: return "background:#dcfce7; color:#166534; font-weight:700;"
        if pct >= 75.0: return "background:#f0fdf4; color:#166534;"
        if pct >= 50.0: return "background:#e0f2fe; color:#075985;"
        if pct >= 25.0: return "background:#fef3c7; color:#92400e;"
        return "background:#ffffff; color:#4b5563;"

html_out = ["""
<script src="https://cdnjs.cloudflare.com/ajax/libs/html2canvas/1.4.1/html2canvas.min.js"></script>
<script>
function downloadElementAsPNG(containerId, scrollId, btnId, filename) {
    const btn = document.getElementById(btnId);
    const container = document.getElementById(containerId);
    const scrollWrap = scrollId ? document.getElementById(scrollId) : null;

    const origText = btn.innerText;
    btn.innerText = 'Rendering 300-DPI PNG...';
    btn.disabled = true;

    let origMaxH = '', origOver = '';
    if (scrollWrap) {
        origMaxH = scrollWrap.style.maxHeight;
        origOver = scrollWrap.style.overflow;
        scrollWrap.style.maxHeight = 'none';
        scrollWrap.style.overflow = 'visible';
    }

    html2canvas(container, {
        backgroundColor: '#ffffff',
        scale: 2.5,
        useCORS: true
    }).then(canvas => {
        if (scrollWrap) {
            scrollWrap.style.maxHeight = origMaxH;
            scrollWrap.style.overflow = origOver;
        }
        const link = document.createElement('a');
        link.download = filename;
        link.href = canvas.toDataURL('image/png');
        link.click();
        btn.innerText = origText;
        btn.disabled = false;
    }).catch(err => {
        console.error(err);
        if (scrollWrap) {
            scrollWrap.style.maxHeight = origMaxH;
            scrollWrap.style.overflow = origOver;
        }
        btn.innerText = 'Export Failed';
        btn.disabled = false;
    });
}
</script>

<style>
    .pub-card {
        background: #ffffff;
        padding: 20px;
        border-radius: 6px;
        border: 1px solid #d1d5db;
        font-family: 'Arial', 'Helvetica Neue', sans-serif;
        color: #111827;
        margin-top: 22px;
        box-shadow: 0 1px 3px rgba(0,0,0,0.05);
    }
    .pub-header-bar {
        display: flex;
        justify-content: space-between;
        align-items: center;
        flex-wrap: wrap;
        gap: 12px;
        margin-bottom: 12px;
        padding-bottom: 10px;
        border-bottom: 2px solid #111827;
    }
    .pub-btn {
        background: #111827;
        color: #ffffff;
        border: 1px solid #111827;
        padding: 7px 14px;
        border-radius: 4px;
        font-size: 12px;
        font-weight: 600;
        cursor: pointer;
    }
    .pub-btn:hover { background: #374151; }
    .booktabs-table {
        border-collapse: collapse;
        width: 100%;
        font-size: 12px;
        font-variant-numeric: tabular-nums;
    }
    .booktabs-table th {
        background: #f8fafc;
        color: #111827;
        padding: 8px 10px;
        text-align: right;
        border-top: 2px solid #111827;
        border-bottom: 1.5px solid #111827;
        font-weight: 700;
    }
    .booktabs-table th:first-child { text-align: left; }
    .booktabs-table td {
        padding: 6px 10px;
        text-align: right;
        border-bottom: 1px solid #e5e7eb;
    }
    .booktabs-table td:first-child {
        text-align: left;
        font-weight: 600;
    }
    .booktabs-total-row td {
        border-top: 1.5px solid #111827;
        border-bottom: 2px solid #111827;
        font-weight: 700;
        background: #f8fafc;
    }
</style>
"""]

# --- TABLE A: CUMULATIVE INTER-ANNOTATOR SD DISTRIBUTION TABLE ---
max_t = SD_THRESHOLDS[-1]
thresh_headers = "".join(
    f"<th>{'= 0f' if t == 0 else f'&le; {t}f'}<br>"
    f"<span style='font-weight:400;font-size:10px;color:#6b7280;'>"
    f"({int(round(t * 1000 / DEFAULT_FPS))} ms)</span></th>"
    for t in SD_THRESHOLDS
)

html_out.append(f"""
<div class="pub-card" id="grand-phase-card">
    <div class="pub-header-bar">
        <div>
            <h3 style="margin:0; font-size:15px; text-transform:uppercase; letter-spacing:0.3px;">
                Table A. Cumulative Distribution of Inter-Annotator SD by Phase (All {total_trials} Trials &amp; {total_unique_reps} Reps Pooled)
            </h3>
            <div style="font-size:12px; color:#4b5563; margin-top:3px;">
                Percentage of repetitions falling within each frame SD threshold (ms equivalents shown at {int(DEFAULT_FPS)} Hz).
            </div>
        </div>
        <button id="btn-dl-phase" class="pub-btn" onclick="downloadElementAsPNG('grand-phase-card', null, 'btn-dl-phase', 'table_sd_distribution_by_phase.png')">
            ⬇ Download SD Distribution (PNG)
        </button>
    </div>
    <table class="booktabs-table">
        <thead>
            <tr>
                <th>Movement Phase</th>
                <th>Reps (N)</th>
                <th>Mean SD (f)</th>
                <th>Median SD (f)</th>
                {thresh_headers}
                <th>&gt; {max_t}f<br><span style='font-weight:400;font-size:10px;color:#6b7280;'>(&gt;{int(round(max_t * 1000 / DEFAULT_FPS))} ms)</span></th>
            </tr>
        </thead>
        <tbody>
""")

for col in SEGMENT_COLS:
    s_f = pooled_sd_df[col].dropna()
    mean_f = s_f.mean()
    med_f  = s_f.median()
    gt_val = (s_f > max_t).mean() * 100.0

    thresh_cells = "".join(
        f"<td style='{get_pct_cell_style((s_f <= t).mean() * 100.0)}'>{(s_f <= t).mean() * 100.0:.1f}%</td>"
        for t in SD_THRESHOLDS
    )

    html_out.append(f"""
        <tr>
            <td>{col}</td>
            <td>{len(s_f)}</td>
            <td style="{get_pub_cell_style(mean_f)}">{mean_f:.2f}</td>
            <td>{med_f:.2f}</td>
            {thresh_cells}
            <td style="{get_pct_cell_style(gt_val, invert=True)}">{gt_val:.1f}%</td>
        </tr>
    """)

tot_thresh_cells = "".join(
    f"<td>{np.mean(all_sd_f_flat <= t) * 100.0:.1f}%</td>"
    for t in SD_THRESHOLDS
)
tot_gt = np.mean(all_sd_f_flat > max_t) * 100.0

html_out.append(f"""
        <tr class="booktabs-total-row">
            <td>All Phases Combined (Grand Total)</td>
            <td>{len(all_sd_f_flat)}</td>
            <td>{np.mean(all_sd_f_flat):.2f}</td>
            <td>{np.median(all_sd_f_flat):.2f}</td>
            {tot_thresh_cells}
            <td style="color:#991b1b;">{tot_gt:.1f}%</td>
        </tr>
        </tbody>
    </table>
</div>
""")

# --- TABLE B: UNIFIED PER-TRIAL SUMMARY TABLE (NO COHORT SEPARATION) ---
html_out.append(f"""
<div class="pub-card" id="unified-trial-card">
    <div class="pub-header-bar">
        <div>
            <h3 style="margin:0; font-size:15px; text-transform:uppercase; letter-spacing:0.3px;">
                Table B. Unified Per-Trial Mean Inter-Annotator SD Across Repetitions ({total_trials} Trials Pooled)
            </h3>
            <div style="font-size:12px; color:#4b5563; margin-top:3px;">
                All trials listed in unified alphabetical order without separating complete vs. fault-tolerant cohorts. Values show trial-level Mean SD in frames (and ms).
            </div>
        </div>
        <button id="btn-dl-trial" class="pub-btn" onclick="downloadElementAsPNG('unified-trial-card', 'unified-trial-scroll', 'btn-dl-trial', 'table_unified_per_trial_summary.png')">
            ⬇ Download Unified Trial Table (PNG)
        </button>
    </div>
    <div id="unified-trial-scroll" style="max-height:600px; overflow:auto; border:1px solid #9ca3af;">
    <table class="booktabs-table" style="width:max-content; min-width:100%;">
        <thead>
            <tr>
                <th style="position:sticky; top:0; z-index:2;">Trial ID</th>
                <th style="position:sticky; top:0; z-index:2;">Reps</th>
                <th style="position:sticky; top:0; z-index:2;">Annotators</th>
                <th style="position:sticky; top:0; z-index:2;">Trial Overall SD (f / ms)</th>
""")

for col in SEGMENT_COLS:
    html_out.append(f"<th style='position:sticky; top:0; z-index:2;'>{col}</th>")
html_out.append("</tr></thead><tbody>")

for e_id in trial_mean_sd_df.index:
    n_r = int(trial_rep_counts[e_id])
    n_a = int(trial_ann_counts[e_id])
    row_sd_f  = trial_mean_sd_df.loc[e_id]
    row_sd_ms = trial_mean_ms_df.loc[e_id]
    overall_trial_f  = row_sd_f.mean()
    overall_trial_ms = row_sd_ms.mean()
    ov_style = get_pub_cell_style(overall_trial_f)

    html_out.append(f"""
        <tr>
            <td>{e_id}</td>
            <td>{n_r}</td>
            <td>{n_a}</td>
            <td style="{ov_style} font-weight:700;">{overall_trial_f:.2f} f ({overall_trial_ms:.0f} ms)</td>
    """)
    for col in SEGMENT_COLS:
        sv = row_sd_f[col]
        mv = row_sd_ms[col]
        cs = get_pub_cell_style(sv)
        cell_str = f"{sv:.1f}f <span style='font-size:10px; opacity:0.8;'>({mv:.0f}ms)</span>" if pd.notna(sv) else "–"
        html_out.append(f"<td style='{cs}'>{cell_str}</td>")
    html_out.append("</tr>")

html_out.append("</tbody></table></div></div>")
display(HTML("".join(html_out)))

✅ Pooled Dataset: 56 Total Trials | 252 Unique Repetitions | 756 Total Annotator Rows


Movement Phase,Reps (N),Mean SD (f),Median SD (f),= 0f(0 ms),≤ 1f(33 ms),≤ 2f(67 ms),≤ 3f(100 ms),≤ 4f(133 ms),≤ 5f(167 ms),≤ 8f(267 ms),> 8f(>267 ms)
start_rest,252,2.63,2.08,23.8%,37.7%,48.8%,56.7%,68.3%,78.2%,100.0%,0.0%
setup,252,2.80,2.52,2.4%,18.3%,38.9%,59.9%,73.8%,86.1%,100.0%,0.0%
first_pull,252,2.12,1.53,4.4%,29.0%,59.5%,75.8%,87.7%,94.0%,100.0%,0.0%
second_pull,252,1.52,1.53,2.8%,37.3%,75.8%,94.8%,98.4%,99.2%,100.0%,0.0%
clean,252,2.26,2.08,1.6%,21.4%,47.2%,73.8%,86.9%,97.6%,100.0%,0.0%
middle_rest,252,3.14,3.06,4.0%,18.7%,32.9%,49.6%,66.7%,80.6%,100.0%,0.0%
jerk_phase_one,252,2.26,2.00,3.6%,21.4%,52.8%,76.6%,87.3%,92.1%,100.0%,0.0%
jerk_phase_two,252,2.85,2.52,4.4%,23.0%,43.3%,58.7%,71.0%,84.9%,100.0%,0.0%
rest_up,252,3.36,3.21,3.2%,15.5%,29.8%,45.2%,60.3%,76.2%,100.0%,0.0%
bring_down,252,2.52,2.20,4.0%,21.0%,40.1%,65.1%,79.8%,91.3%,100.0%,0.0%


In [ ]:
# @title
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ==============================================================================
# CONFIGURATION
# ==============================================================================
ENTRY_TYPES = ['complete_trials', 'fault_tolerant_trials'] #[cite: 1]
CANDIDATE_TAGS = [f'rep_segments_a{i}' for i in range(1, 16)] # Scans a1 through a15

# Annotator ID -> Name mapping from your Wearable Dataset Tracker sheet
ANNOTATOR_NAMES = {
    1: "Thevindu", 2: "Janith", 3: "Praveen", 4: "Iresh", 5: "Chamodya",
    6: "Sakith", 7: "Oshan", 8: "Devni", 9: "Ridma", 10: "Udamgie",
    11: "Isuri", 12: "Hansika", 13: "Sithila"
} #

SEGMENT_COLS = [
    'start_rest', 'setup', 'first_pull', 'second_pull', 'clean',
    'middle_rest', 'jerk_phase_one', 'jerk_phase_two', 'rest_up',
    'bring_down', 'end_rest', 'end_rep'
] #[cite: 1]

# ==============================================================================
# 1. SCAN OPENDATASETSTORE FOR AVAILABLE ANNOTATIONS PER TRIAL
# ==============================================================================
audit_rows = []

for e_type in ENTRY_TYPES:
    entries_df = store.list_entries(entry_type=e_type) #[cite: 1]
    if entries_df is None or entries_df.empty:
        continue

    for _, meta in entries_df.iterrows():
        e_id = meta['entry_id'] #[cite: 1]
        sub_id = meta.get('entity_id', '') #[cite: 1]

        found_details = []
        rep_counts = []
        empty_files = []

        for tag in CANDIDATE_TAGS:
            try:
                df_ann = store.get_entry_processed_data(entry_type=e_type, entry_id=e_id, tag=tag) #[cite: 1]
                if df_ann is not None:
                    ann_num = int(tag.split('_a')[-1])
                    ann_name = ANNOTATOR_NAMES.get(ann_num, f"ID_{ann_num}")

                    if df_ann.empty:
                        empty_files.append(f"a{ann_num} ({ann_name})")
                        continue

                    # Check valid non-zero reps[cite: 1]
                    valid_mask = (df_ann[SEGMENT_COLS[1:]] != 0).any(axis=1) #[cite: 1]
                    n_valid_reps = int(valid_mask.sum())

                    if n_valid_reps > 0:
                        found_details.append(f"a{ann_num}: {ann_name} ({n_valid_reps}r)")
                        rep_counts.append(n_valid_reps)
                    else:
                        empty_files.append(f"a{ann_num} ({ann_name} - all 0s)")
            except Exception:
                continue

        n_found = len(found_details)
        rep_mismatch = len(set(rep_counts)) > 1

        # Determine Status
        if n_found == 3 and not rep_mismatch and not empty_files:
            status = "✅ Complete (3/3)"
        elif n_found == 3 and rep_mismatch:
            status = "⚠️ Rep Count Mismatch"
        elif n_found > 3:
            status = f"🟣 Extra Files ({n_found}/3)"
        elif n_found == 0:
            status = "❌ Missing All (0/3)"
        else:
            status = f"🔴 Incomplete ({n_found}/3)"

        audit_rows.append({
            'entry_type': e_type,
            'entry_id': e_id,
            'subject': sub_id,
            'status': status,
            'count': n_found,
            'rep_counts': str(rep_counts) if rep_counts else "-",
            'available_annotators': "  |  ".join(found_details) if found_details else "None",
            'empty_or_corrupt_files': ", ".join(empty_files) if empty_files else "-"
        })

audit_df = pd.DataFrame(audit_rows)

# ==============================================================================
# 2. SUMMARY COUNTS & STYLED AVAILABILITY TABLE
# ==============================================================================
print("=== 📊 OVERALL ANNOTATION AVAILABILITY SUMMARY ===")
summary_tbl = pd.crosstab(audit_df['entry_type'], audit_df['status'], margins=True)
display(summary_tbl)

def style_status(val):
    if "✅" in val:
        return "background-color: #1e4620; color: #a3e635; font-weight: bold;"
    elif "⚠️" in val or "🟣" in val:
        return "background-color: #5c4300; color: #fde047; font-weight: bold;"
    else:
        return "background-color: #6b1414; color: #fca5a5; font-weight: bold;"

print("\n=== 📋 DETAILED PER-EXPERIMENT ANNOTATION INVENTORY ===")
styled_audit = (
    audit_df.style
    .map(style_status, subset=['status'])
    .set_properties(**{'text-align': 'left', 'font-family': 'monospace', 'font-size': '12px'})
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
)
display(styled_audit)